In [1]:
import pandas as pd 
import numpy as np 

In [4]:
desempleo = pd.read_excel('Data\\Desempleo.xlsx')
dolar = pd.read_excel('Data\\Dolar.xlsx')
inflacion = pd.read_excel('Data\\Inflacion.xlsx')
tbills = pd.read_excel('Data\\Tbills 5 años.xlsx')
tes = pd.read_excel('Data\\TES 5 años.xlsx')
tpm = pd.read_excel('Data\\TPM.xlsx')

bases = [desempleo , dolar , inflacion , tbills , tes , tpm]

In [6]:
for i in bases:
    print(i.columns)
    print()
    print(i.shape)

 

Index(['Fecha', 'Desempleo '], dtype='object')

(265, 2)
Index(['Fecha', 'TRM'], dtype='object')

(8155, 2)
Index(['Fecha', 'Inflación total'], dtype='object')

(267, 2)
Index(['Fecha', 'T Bills'], dtype='object')

(5823, 2)
Index(['Fecha', 'TES '], dtype='object')

(5429, 2)
Index(['Fecha', 'Tasa de política monetaria'], dtype='object')

(8154, 2)


In [18]:
colmap = {
    "Desempleo ": "desempleo",
    "TRM": "trm",
    "Inflación total": "inflacion",
    "T Bills": "tbills",
    "TES ": "tes",
    "Tasa de política monetaria": "tpm"}


def to_quarterly_mean(df, date_col, value_col, out_name):
    """
    Promedia a frecuencia trimestral (fin de trimestre) y devuelve DataFrame con esa única columna.
    """
    df = df.rename(columns=lambda c: c.strip())
    date_col = date_col.strip()
    value_col = value_col.strip()

    # Parseo y limpieza
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = (df.dropna(subset=[date_col, value_col])
            .sort_values(date_col)
            .drop_duplicates(subset=[date_col], keep="last"))

    s = df.set_index(date_col)[value_col].astype(float)
    qs = s.resample("QE").mean().to_frame(out_name)
    return qs

q_desempleo = to_quarterly_mean(desempleo, "Fecha", "Desempleo ", colmap["Desempleo "])
q_desempleo = to_quarterly_mean(desempleo, "Fecha", "Desempleo ", colmap["Desempleo "])
q_infl = to_quarterly_mean(inflacion, "Fecha", "Inflación total", colmap["Inflación total"])
q_tbills = to_quarterly_mean(tbills,    "Fecha", "T Bills",       colmap["T Bills"])
q_tes = to_quarterly_mean(tes,       "Fecha", "TES ",          colmap["TES "])
q_tpm  = to_quarterly_mean(tpm,       "Fecha", "Tasa de política monetaria", colmap["Tasa de política monetaria"])

# Limpieza de TRM diaria (o irregular) y selección de fin de trimestre
trm = dolar.rename(columns=lambda c: c.strip())
trm["Fecha"] = pd.to_datetime(trm["Fecha"], errors="coerce")
trm = (trm.dropna(subset=["Fecha", "TRM"])
          .sort_values("Fecha")
          .drop_duplicates(subset=["Fecha"], keep="last")
          .set_index("Fecha"))

trm_q_eop = trm["TRM"].astype(float).resample("QE").last().to_frame("trm_eop")
dlog_trm_q = np.log(trm_q_eop["trm_eop"]).diff().to_frame("dlog_trm")

df_q = pd.concat(
    [q_desempleo, q_infl, q_tbills, q_tes, q_tpm, dlog_trm_q],
    axis=1, join="inner").sort_index().dropna()

print("Rango común trimestral:", df_q.index.min().date(), "→", df_q.index.max().date())
print("Número de trimestres:", len(df_q))
print("Columnas:", list(df_q.columns))
print(df_q.tail())


Rango común trimestral: 2003-06-30 → 2025-03-31
Número de trimestres: 88
Columnas: ['desempleo', 'inflacion', 'tbills', 'tes', 'tpm', 'dlog_trm']
            desempleo  inflacion    tbills        tes        tpm  dlog_trm
Fecha                                                                     
2024-03-31  11.873333   7.816667  4.121967   9.370000  12.802198  0.005284
2024-06-30  10.460000   7.166667  4.464286  10.262623  11.920330  0.076565
2024-09-30   9.570000   6.263333  3.799531   9.722857  10.923913  0.003891
2024-12-31   8.810000   5.270000  4.123387  10.027377   9.895604  0.057155
2025-03-31  10.985000   5.196667  4.250000  10.801475   9.500000 -0.050368


In [19]:
df_q

,desempleo,inflacion,tbills,tes,tpm,dlog_trm
Fecha,,,,,,
2003-06-30,13.980000,7.596667,2.570000,15.066271,6.942308,-0.048812
2003-09-30,14.373333,7.136667,3.138125,14.378281,7.250000,0.025259
2003-12-31,12.906667,6.400000,3.244677,14.185167,7.250000,-0.039239
2004-03-31,15.510000,6.226667,2.980968,12.878710,7.120879,-0.036677
2004-06-30,14.193333,5.643333,3.720968,13.281333,6.750000,0.007966
...,...,...,...,...,...,...
2024-03-31,11.873333,7.816667,4.121967,9.370000,12.802198,0.005284
2024-06-30,10.460000,7.166667,4.464286,10.262623,11.920330,0.076565
2024-09-30,9.570000,6.263333,3.799531,9.722857,10.923913,0.003891


In [20]:
df_q.to_csv('Base_Completa.csv')